In [5]:
%pip install pymannkendall

Defaulting to user installation because normal site-packages is not writeable
  Using cached pymannkendall-1.4.3-py3-none-any.whl.metadata (14 kB)
Using cached pymannkendall-1.4.3-py3-none-any.whl (12 kB)

[notice] A new release of pip is available: 25.1 -> 25.1.1
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
import pandas as pd
import requests
import os
import glob
import datetime as datetime
from dataIO import dataloader, webservices
# from dataIO.webservices import fetch_ca_water_data
from statisticscalculator import generalstatistics, climatestatistics
from plot_collection import stackedlineplots, streamflow_stat_plots, streamflow_stat_plots_matplotlib
import plotly.express as px
from plotly import graph_objects as go

In [7]:
import io

def fetch_ca_water_data(start_date, end_date, stations, parameters):
    base_url = 'https://wateroffice.ec.gc.ca/services/daily_data/csv/inline'
    
    # Format station and parameter lists for URL query parameters
    station_params = '&'.join([f'stations[]={station}' for station in stations])
    parameter_params = '&'.join([f'parameters[]={parameter}' for parameter in parameters])
    
    # Construct the complete URL with parameters
    url = f'{base_url}?{station_params}&{parameter_params}&start_date={start_date}&end_date={end_date}'
    
    # Send HTTP GET request to fetch data
    response = requests.get(url)
    
    # Check if request was successful
    if response.status_code == 200:
        # Read CSV data directly from response content
        data = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
        return data
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        return None

In [3]:
headwater = pd.read_excel('headwater_sites_revised.xlsx')
headwater.set_index('site_no', inplace=True)
site_list = list(headwater.index)
lats = list(headwater['lat'])
longs = list(headwater['long']) 


all_site_data = {}
for site in site_list:
    try:
        # basin = site[1][1]
        data = webservices.usgs_streamflow().get_data(start_date='1968-10-01', end_date='2023-10-01',sites=str(site)).reset_index()
        all_site_data[site] = data
    except Exception as e:
        print(e)
    finally:
        pass

https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12301250&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12302055&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12304500&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12306500&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12324400&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12324590&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12332000&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=j

In [8]:
import sqlite3
conn = sqlite3.connect('BC Data/Hydat.sqlite3')
cursor = conn.cursor()
sql_statement = "SELECT * FROM sqlite_master WHERE type='table';"
cursor.execute(sql_statement)

# To see tables in the db
tables = cursor.fetchall()
tablez = []
for table in tables:
    tablez.append(table[2])
print(f'TABLES IN DB: {tablez}')
# To see the column names of a specific table:
query = "PRAGMA table_info(STATIONS)";
cursor.execute(query) 
print(f'COLUMNS IN TABLE STATIONS: {cursor.fetchall()}')
    
canadian_stations = [
    '08LD001', '07EA007', '08NL004','09AA006','08FB006','08EC001','08EC013','08KE016','08LB020','08LB069','07FC001',
    '08FB007','08MB006','08NE039','08NB012','08LB038','07FC003','10CD004','08KD007','08EE013','08NN002','07FA005',
    '08ND012','08NB014','08NH130','08KB001','08KA007','08KA005','08KA004','08NK018','08NP001','07FB009','07EA005',
    '08EG012','08NK016','08NK002','08GA071','08LE024','08NH119','08JD006','08EE004','08EE003','08NC004','08GA072',
    '08MG001','08HA001','08MB005','08MA002','08MA001','08MH103','08MH016','08MH001','07EE009','08LA001','08LG048',
    '08NB005','08NA002','10AC005','08FC003','08KA001','08JD006','08NH119','08LE024','08GA071','08NK002','08NK016',
    '08EG012','07EA005','07FB009','08NP001','08NK018','08KA004','08KA005','08KA007','08KB001','08NH130','08NB014',
    '08ND012','07FA005','08NN002'
]
parameters = ['flow']

all_ca_site_data = fetch_ca_water_data(start_date='1968-10-01', end_date='2023-10-01', stations=canadian_stations, parameters=parameters)


query = """
    SELECT STATION_NUMBER, STATION_NAME, PROV_TERR_STATE_LOC, LATITUDE, LONGITUDE, 
    DRAINAGE_AREA_GROSS FROM STATIONS 
    WHERE STATION_NUMBER IN ({})
""".format(','.join(['?']*len(canadian_stations)))

cursor.execute(query, canadian_stations)
station_meta_data = cursor.fetchall()
print(len(canadian_stations))
station_meta_data=pd.DataFrame(station_meta_data)
station_meta_data.columns = ['STATION_NUMBER', 'STATION_NAME', 'PROV_TERR_STATE_LOC', 'LATITUDE', 
                             'LONGITUDE', 'DRAINAGE_AREA_GROSS']
station_meta_data
# if all_ca_site_data is not None:
#     all_ca_site_data.head() 

TABLES IN DB: ['STATIONS', 'CONCENTRATION_SYMBOLS', 'SED_SAMPLES_PSD', 'ANNUAL_INSTANT_PEAKS', 'STN_DATUM_UNRELATED', 'DATA_SYMBOLS', 'SED_VERTICAL_LOCATION', 'STN_DATA_COLLECTION', 'PEAK_CODES', 'SED_DATA_TYPES', 'MEASUREMENT_CODES', 'SED_VERTICAL_SYMBOLS', 'DATA_TYPES', 'DLY_FLOWS', 'STN_REMARKS', 'STN_DATUM_CONVERSION', 'AGENCY_LIST', 'SED_DLY_SUSCON', 'STN_OPERATION_SCHEDULE', 'STN_DATA_RANGE', 'PRECISION_CODES', 'SED_DLY_LOADS', 'DLY_LEVELS', 'OPERATION_CODES', 'STN_REGULATION', 'DATUM_LIST', 'ANNUAL_STATISTICS', 'VERSION', 'REGIONAL_OFFICE_LIST', 'SAMPLE_REMARK_CODES', 'STN_REMARK_CODES', 'SED_SAMPLES', 'STN_STATUS_CODES']
COLUMNS IN TABLE STATIONS: [(0, 'STATION_NUMBER', 'TEXT', 0, None, 0), (1, 'STATION_NAME', 'TEXT', 0, None, 0), (2, 'PROV_TERR_STATE_LOC', 'TEXT', 0, None, 0), (3, 'REGIONAL_OFFICE_ID', 'TEXT', 0, None, 0), (4, 'HYD_STATUS', 'TEXT', 0, None, 0), (5, 'SED_STATUS', 'TEXT', 0, None, 0), (6, 'LATITUDE', 'DOUBLE', 0, None, 0), (7, 'LONGITUDE', 'DOUBLE', 0, None, 0),

,STATION_NUMBER,STATION_NAME,PROV_TERR_STATE_LOC,LATITUDE,LONGITUDE,DRAINAGE_AREA_GROSS
0,07EA005,FINLAY RIVER ABOVE AKIE RIVER,BC,57.126221,-125.249924,15600.0
1,07EA007,AKIE RIVER NEAR THE 760 M CONTOUR,BC,57.191109,-124.901108,1690.0
2,07EE009,CHUCHINKA CREEK NEAR THE MOUTH,BC,54.529720,-122.612221,310.0
3,07FA005,GRAHAM RIVER ABOVE COLT CREEK,BC,56.459862,-122.354530,2140.0
4,07FB009,FLATBED CREEK AT KILOMETRE 110 HERITAGE HIGHWAY,BC,55.090172,-120.941833,486.0
5,07FC001,BEATTON RIVER NEAR FORT ST. JOHN,BC,56.278439,-120.699860,15600.0
6,07FC003,BLUEBERRY RIVER BELOW AITKEN CREEK,BC,56.677639,-121.222328,1770.0
7,08EC001,BABINE RIVER AT BABINE,BC,55.322529,-126.629829,6350.0
8,08EC013,BABINE RIVER AT OUTLET OF NILKITKWA LAKE,BC,55.426540,-126.697632,6760.0
9,08EE003,BULKLEY RIVER NEAR HOUSTON,BC,54.399380,-126.719414,2370.0


In [9]:
len(tablez)

33

In [10]:
station_meta_data

,STATION_NUMBER,STATION_NAME,PROV_TERR_STATE_LOC,LATITUDE,LONGITUDE,DRAINAGE_AREA_GROSS
0,07EA005,FINLAY RIVER ABOVE AKIE RIVER,BC,57.126221,-125.249924,15600.0
1,07EA007,AKIE RIVER NEAR THE 760 M CONTOUR,BC,57.191109,-124.901108,1690.0
2,07EE009,CHUCHINKA CREEK NEAR THE MOUTH,BC,54.529720,-122.612221,310.0
3,07FA005,GRAHAM RIVER ABOVE COLT CREEK,BC,56.459862,-122.354530,2140.0
4,07FB009,FLATBED CREEK AT KILOMETRE 110 HERITAGE HIGHWAY,BC,55.090172,-120.941833,486.0
5,07FC001,BEATTON RIVER NEAR FORT ST. JOHN,BC,56.278439,-120.699860,15600.0
6,07FC003,BLUEBERRY RIVER BELOW AITKEN CREEK,BC,56.677639,-121.222328,1770.0
7,08EC001,BABINE RIVER AT BABINE,BC,55.322529,-126.629829,6350.0
8,08EC013,BABINE RIVER AT OUTLET OF NILKITKWA LAKE,BC,55.426540,-126.697632,6760.0
9,08EE003,BULKLEY RIVER NEAR HOUSTON,BC,54.399380,-126.719414,2370.0


In [11]:
mf_flows = pd.read_excel('mf_flows_appendix_a_w_loc.xlsx')
mf_flows.rename(columns={'Lat':'Long', 'Long':'Lat'}, inplace=True)
mf_flows.set_index('ID', inplace=True) # chaining .set_index() apparently doesn't work.
mf_flows.head(3)

,Name,Basin,Long,Lat,Section,H,S,A,L,ARF,D,DD,E,EE,M
ID,,,,,,,,,,,,,,,
ALF,Albeni Falls,Pend Oreille,-117.030000,48.180000,3.2,x,x,x,x,x,x,x,NaN,NaN,NaN
ANA,Anatone,Lower Snake,-116.976667,46.097222,3.5,x,x,x,x,x,NaN,NaN,NaN,NaN,NaN
BIT,Bitterroot,Pend Oreille,-114.050000,46.830000,3.2,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
modified_flows_m = {}

for file in glob.glob('modifiedflows2020/m_files/*'):
    # print(file)
    filename = file.split('/')[2]
    sitename = filename.split('_')[0]
    # print(sitename[:3])
    # print(mf_flows['ID'])
    if sitename[:3] in mf_flows.index.to_list():
        df = pd.read_excel(file)
        # print(sitename[:3])
        modified_flows_m[sitename] = df
# print(len(modified_flows_m.keys()))

In [ ]:
# a = str(pd.to_datetime('1960-10-01').date())
# b = str(pd.to_datetime('2018-10-10').date())
# f_ca[f_ca['Date'].between(a, b)]

In [33]:
d = dataloader.DataLoader(filtered_all_ca_site_data.get('07EE009'), 'Date', 'Value/Valeur')

In [56]:
filtered_df = all_site_data.get(site)[all_site_data.get(site)['Date']].between(start_date, end_date)


TypeError: 'NoneType' object is not subscriptable

In [7]:
end_date = '2023-10-01'
start_dates = [
#     '1938-10-01', 
#     '1948-10-01', 
    '1968-10-01',
#     '1975-10-01',
    # '1978-10-01',
    # '1988-10-01',
    # '1998-10-01', 
#     '2010-10-01', 
#     '2018-10-01', 
]

for start_date in start_dates:
# for start_date, end_date in zip(start_dates, end_dates):

###-------USGS gages/Headwater sites-----------------------------
#     file_name = f'Headwater_MK_Results_{start_date}to{end_date}.html'
    # try:
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()
    filtered_all_site_data = {}
    yrs_of_data = []
    included_sites = []
    excluded_sites = []
    for site, df in all_site_data.items():
        # num_yrs_of_data = len(df['Date'])/365
        # yrs_of_data.append(num_yrs_of_data)
    
        filtered_df = all_site_data.get(site)[all_site_data.get(site)['Date'].between(start_date, end_date)]
        #if more than 15% of data is missing during the time period, exclude station
        if ((end_date - start_date).days * 0.85) > len(filtered_df['Discharge']):
            print(f"Site {site} is missing more than 15% of data between {start_date} and {end_date}.  It will NOT be included in the study.")
            excluded_sites.append(site)
        else:
            filtered_all_site_data[site] = filtered_df
            print(f'Site {site} has more than 85% of data between {start_date} and {end_date} and will be included in the study.')
            included_sites.append(site)
    
    volume_stats_hw = {}
    print(f'Number of included headwater sites: {len(included_sites)}')
    print(f'Number of headwater sites excluded from study: {len(excluded_sites)}')
    
    for site, site_df in filtered_all_site_data.items():
        d = dataloader.DataLoader(site_df, 'Date', name_of_Q_column='Discharge')
        s = climatestatistics.Streamflow(d)
      
        try:
            mann_kendall_results_hw = {}
            mann_kendall_results_hw['lat'] = headwater[headwater.index==site].iat[0,1]
            mann_kendall_results_hw['long'] = headwater[headwater.index==site].iat[0,2]
            mann_kendall_results_hw['name'] = headwater[headwater.index==site].iat[0,0]
            s.calc_max(calc_from_rolling_median=False, window_size=7)
            s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
            s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
            mann_kendall_results_hw['Fall Vol']=s.volume_bw_days_mann_kendall_test
            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
#             print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
            mann_kendall_results_hw['Winter Vol']=s.volume_bw_days_mann_kendall_test
#             print('-------------------------------------------------')
        #     for day in ['01-01', '03-01', '05-01', '08-01']:
            s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
        #     runoff_bw_dates_plot2 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
        #     runoff_bw_dates_plot2.plot_runoff_volume_between_2days(site[0])
            mann_kendall_results_hw['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
#             print('-------------------------------------------------')            
        #     for day in ['01-01', '03-01', '05-01', '08-01']:
            s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
        #     runoff_bw_dates_plot3 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            mann_kendall_results_hw['Summer Vol'] = s.volume_bw_days_mann_kendall_test

        #     runoff_bw_dates_plot3.plot_runoff_volume_between_2days(site[0])


            mann_kendall_results_hw['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
            mann_kendall_results_hw['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
            mann_kendall_results_hw['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
            mann_kendall_results_hw['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
            mann_kendall_results_hw['Total Vol MK Test'] = s.total_volume_mann_kendall_test
            mann_kendall_results_hw['Site Type'] = 'headwater'
            volume_stats_hw[site] = mann_kendall_results_hw

        except Exception as e:
            raise ValueError('Saving to volume_stats_hw prob failed')
            print(e)

##-----------Canadian sites---------------            
    # try:
#     start_date = pd.to_datetime(start_date).date()
#     end_date = pd.to_datetime(end_date).date()
    filtered_all_ca_site_data = {}
    yrs_of_data = []
    included_sites = []
    excluded_sites = []
    for site in all_ca_site_data[' ID'].unique(): #station_meta_data['STATION_NUMBER'].unique():
        # num_yrs_of_data = len(df['Date'])/365
        # yrs_of_data.append(num_yrs_of_data)
    #fiiiiiiii
        filtered_df = all_ca_site_data[all_ca_site_data[' ID']==site]
        f_ca = filtered_df
#         pd.to_datetime(filtered_df['Date'])
        filtered_df = filtered_df[filtered_df.loc[:, 'Date'].between(str(start_date), str(end_date))]
#         print(type(filtered_df['Date']))
#         filtered_df.loc[:, 'Date'] = pd.to_datetime(filtered_df.loc[:, 'Date'])
        #if more than 15% of data is missing during the time period, exclude station
        if ((end_date - start_date).days * 0.85) > len(filtered_df['Value/Valeur']):
            print(f"Site {site} is missing more than 15% of data between {start_date} and {end_date}.  It will NOT be included in the study.")
            excluded_sites.append(site)
        else:
            filtered_all_ca_site_data[site] = filtered_df
            print(f'Site {site} has more than 85% of data between {start_date} and {end_date} and will be included in the study.')
            included_sites.append(site)
    
    volume_stats_ca = {}
#     print(f'Number of included headwater sites: {len(included_sites)}')
#     print(f'Number of headwater sites excluded from study: {len(excluded_sites)}')
    
    for site in all_ca_site_data[' ID'].unique(): #station_meta_data['STATION_NUMBER'].unique():
        print(site)
        try:
            d = dataloader.DataLoader(filtered_all_ca_site_data.get(site), 'Date', 'Value/Valeur')
            s = climatestatistics.Streamflow(d)
#         except:
#             pass
#         try:
            mann_kendall_results_ca = {}
            mann_kendall_results_ca['lat'] = station_meta_data[station_meta_data['STATION_NUMBER']==site]['LATITUDE']
            mann_kendall_results_ca['long'] = station_meta_data[station_meta_data['STATION_NUMBER']==site]['LONGITUDE']
            mann_kendall_results_ca['name'] = station_meta_data[station_meta_data['STATION_NUMBER']==site]['STATION_NAME']
#             s.calc_max(calc_from_rolling_median=False, window_size=7)
#             s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
#             s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
#             # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#             runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
#             mann_kendall_results_ca['Fall Vol']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
# #             print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#             runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
#             mann_kendall_results_ca['Winter Vol']=s.volume_bw_days_mann_kendall_test
# #             print('-------------------------------------------------')
#         #     for day in ['01-01', '03-01', '05-01', '08-01']:
#             s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)
#             # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#         #     runoff_bw_dates_plot2 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#         #     runoff_bw_dates_plot2.plot_runoff_volume_between_2days(site[0])
#             mann_kendall_results_ca['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
# #             print('-------------------------------------------------')            
#         #     for day in ['01-01', '03-01', '05-01', '08-01']:
#             s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)
#             # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#         #     runoff_bw_dates_plot3 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             mann_kendall_results_ca['Summer Vol'] = s.volume_bw_days_mann_kendall_test

#         #     runoff_bw_dates_plot3.plot_runoff_volume_between_2days(site[0])
#             mann_kendall_results_ca['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
#             mann_kendall_results_ca['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
#             mann_kendall_results_ca['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
#             mann_kendall_results_ca['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
#             mann_kendall_results_ca['Total Vol MK Test'] = s.total_volume_mann_kendall_test
#             mann_kendall_results_ca['Site Type'] = 'canadian'
#             volume_stats_ca[site] = mann_kendall_results_ca

#         except Exception as e:
#             raise ValueError('Saving to volume_stats_ca prob failed')
#             print(e)    
#feeeeeeee
            s.calc_max(calc_from_rolling_median=False, window_size=7)  
            s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
            s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            mann_kendall_results_ca['Fall Vol']=s.volume_bw_days_mann_kendall_test

            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            mann_kendall_results_ca['Winter Vol']=s.volume_bw_days_mann_kendall_test

            s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)

            mann_kendall_results_ca['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test

            s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)

            mann_kendall_results_ca['Summer Vol'] = s.volume_bw_days_mann_kendall_test
           
            mann_kendall_results_ca['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
            mann_kendall_results_ca['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
            mann_kendall_results_ca['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
            mann_kendall_results_ca['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
            mann_kendall_results_ca['Total Vol MK Test'] = s.total_volume_mann_kendall_test
            mann_kendall_results_ca['Site Type'] = 'canadian'
            volume_stats_ca[site] = mann_kendall_results_ca
            
        except Exception as e:
#             raise ValueError('Saving to volume_stats_ca prob failed')
            print('Saving to volume_stats_ca prob failed')
            print(e)
            pass
            
            
            
            
            
# -------Modified Flows----------

#     modified_flows_m_filtered_dates = {}
#     for k,v in modified_flows_m.items():
# #         print(f'dates: {start_date} and {end_date}')
# #         print(f'date types: {type(start_date)} and {type(end_date)}')
# #         start_date = pd.to_datetime(start_date).date()
# #         end_date = pd.to_datetime(end_date).date()
# #         print(f'dates: {start_date} and {end_date}')
# #         print(f'date types: {type(start_date)} and {type(end_date)}')
# #         print(f"type of df data: {type(modified_flows_m.get(k)['date'])}")
        
#         start_date = str(start_date)
#         end_date = str(end_date)
        
#         # modified_flows_m_filtered_dates[k] = modified_flows_m.get(k)[modified_flows_m.get(k)['date']>=start_date]
#         modified_flows_m_filtered_dates[k] = modified_flows_m.get(k)[modified_flows_m.get(k)['date'].between(start_date, end_date)]
    
    
    
#     volume_stats_mf = {}    
#     for site in modified_flows_m_filtered_dates.items():
# #         print(site)
#         sitename = site[0].split('_')[0]    
#         # print(sitename)
#         stupid = sitename.split('6')[1]
#         # print(stupid)
#         name_of_Q_column = f'{stupid} (unit:cfs)'
#         # print(name_of_Q_column)
#     #     data = webservices.usgs_streamflow().get_data(start_date='1921-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=str(site[1])).reset_index()
#         d = dataloader.DataLoader(site[1], 'date', name_of_Q_column=name_of_Q_column)
#         s = climatestatistics.Streamflow(d)
#         # print(d._df[d._name_of_Q_column][0])
#         site_short = site[0][:3]
#         # print(f'Site: {site_short}')
    
#         try:
#             mann_kendall_results_mf = {}
         
#             s.calc_max(calc_from_rolling_median=False, window_size=7)  
#             s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
#             s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            
#             runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             mann_kendall_results_mf['Fall Vol']=s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
#             runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             mann_kendall_results_mf['Winter Vol']=s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)

#             mann_kendall_results_mf['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)

#             mann_kendall_results_mf['Summer Vol'] = s.volume_bw_days_mann_kendall_test
           
            
#             mann_kendall_results_mf['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
#             mann_kendall_results_mf['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
#             mann_kendall_results_mf['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
#             mann_kendall_results_mf['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
#             mann_kendall_results_mf['Total Vol MK Test'] = s.total_volume_mann_kendall_test
#             # mann_kendall_results['Basin'] = basin
#             mann_kendall_results_mf['lat'] = mf_flows[mf_flows.index==site_short].iat[0,3]
#             mann_kendall_results_mf['long'] = mf_flows[mf_flows.index==site_short].iat[0,2]
#             # print(mann_kendall_results['long'])
#             mann_kendall_results_mf['name'] = mf_flows[mf_flows.index==site_short].iat[0,0]
#             volume_stats_mf[site_short] = mann_kendall_results_mf
#             mann_kendall_results_mf['Site Type'] = 'modified flows'

#         except Exception as e:
#             raise ValueError('Saving to volume_stats_mf prob failed')
#             print(e)
 
    
    
    
    sites = []
    site_name = []
    site_type = []
    lat = []
    long = []
    # basins = []
    
    PeakRunoffQ_trend = []
    PeakRunoffQ_h = []
    PeakRunoffQ_p = []
    PeakRunoffQ_tau = []
    
    PeakRunoffDOY_trend = []
    PeakRunoffDOY_h = []
    PeakRunoffDOY_p = []
    PeakRunoffDOY_tau = []
    
    ThresholdVol_trend = []
    ThresholdVol_h = []
    ThresholdVol_p = []
    ThresholdVol_tau = []
    
    ThresholdDOY_trend = []
    ThresholdDOY_p = []
    ThresholdDOY_tau = []
    
    ThresholdQ_trend = []
    ThresholdQ_h = []
    ThresholdQ_p = []
    ThresholdQ_tau = []
    
    TotalVol_trend = []
    TotalVol_h = []
    TotalVol_p = []
    TotalVol_tau = []
    
    RunoffVol_trend = []
    RunoffVol_h = []
    RunoffVol_p = []
    RunoffVol_tau = []
    
    SummerVol_trend = []
    SummerVol_h = []
    SummerVol_p = []
    SummerVol_tau = []
    
    FallVol_trend = []
    FallVol_h = []
    FallVol_p = []
    FallVol_tau = []
    
    WinterVol_trend = []
    WinterVol_h = []
    WinterVol_p = []
    WinterVol_tau = []
    
    for site, value in volume_stats_hw.items():
        
        sites.append(site)
        site_name.append(value.get('name'))
        site_type.append('headwater')
        lat.append(value.get('lat'))
        long.append(value.get('long'))              
                         
        # basins.append(site[1]['Basin'])
        
        PeakRunoffQ_trend.append(value.get('Q on date of peak Q date MK Test')[0])
        PeakRunoffQ_p.append(value.get('Q on date of peak Q date MK Test')[2])
        PeakRunoffQ_tau.append(value.get('Q on date of peak Q date MK Test')[4])
        
        PeakRunoffDOY_trend.append(value.get('Q on date of peak Q date MK Test')[0])
        PeakRunoffDOY_p.append(value.get('Q on date of peak Q date MK Test')[2])
        PeakRunoffDOY_tau.append(value.get('Q on date of peak Q date MK Test')[4])    
            
        ThresholdVol_trend.append(value.get('Threshold Vol MK Test')[0])
        ThresholdVol_p.append(value.get('Threshold Vol MK Test')[2])
        ThresholdVol_tau.append(value.get('Threshold Vol MK Test')[4])    
            
        ThresholdDOY_trend.append(value.get('Threshold Vol DOY MK Test')[0])
        ThresholdDOY_p.append(value.get('Threshold Vol DOY MK Test')[2])
        ThresholdDOY_tau.append(value.get('Threshold Vol DOY MK Test')[4])            
            
        TotalVol_trend.append(value.get('Total Vol MK Test')[0])
        TotalVol_p.append(value.get('Total Vol MK Test')[2])
        TotalVol_tau.append(value.get('Total Vol MK Test')[4])
        
        SummerVol_trend.append(value.get('Summer Vol')[0])
        SummerVol_p.append(value.get('Summer Vol')[2])
        SummerVol_tau.append(value.get('Summer Vol')[4])
    
        RunoffVol_trend.append(value.get('Runoff Season Vol')[0])
        RunoffVol_p.append(value.get('Runoff Season Vol')[2])
        RunoffVol_tau.append(value.get('Runoff Season Vol')[4])
        
        WinterVol_trend.append(value.get('Winter Vol')[0])
        WinterVol_p.append(value.get('Winter Vol')[2])
        WinterVol_tau.append(value.get('Winter Vol')[4])
    
        FallVol_trend.append(value.get('Fall Vol')[0])
        FallVol_p.append(value.get('Fall Vol')[2])
        FallVol_tau.append(value.get('Fall Vol')[4])
    
    for site, value in volume_stats_ca.items():
        
        sites.append(site)
        site_name.append(value.get('name'))
        site_type.append('canadian')
        lat.append(value.get('lat'))
        long.append(value.get('long'))              
                         
        # basins.append(site[1]['Basin'])
        
#         PeakRunoffQ_trend.append(value.get('Q on date of peak Q date MK Test')[0])
#         PeakRunoffQ_p.append(value.get('Q on date of peak Q date MK Test')[2])
#         PeakRunoffQ_tau.append(value.get('Q on date of peak Q date MK Test')[4])
        
#         PeakRunoffDOY_trend.append(value.get('Q on date of peak Q date MK Test')[0])
#         PeakRunoffDOY_p.append(value.get('Q on date of peak Q date MK Test')[2])
#         PeakRunoffDOY_tau.append(value.get('Q on date of peak Q date MK Test')[4])    
            
#         ThresholdVol_trend.append(value.get('Threshold Vol MK Test')[0])
#         ThresholdVol_p.append(value.get('Threshold Vol MK Test')[2])
#         ThresholdVol_tau.append(value.get('Threshold Vol MK Test')[4])    
            
#         ThresholdDOY_trend.append(value.get('Threshold Vol DOY MK Test')[0])
#         ThresholdDOY_p.append(value.get('Threshold Vol DOY MK Test')[2])
#         ThresholdDOY_tau.append(value.get('Threshold Vol DOY MK Test')[4])            
            
#         TotalVol_trend.append(value.get('Total Vol MK Test')[0])
#         TotalVol_p.append(value.get('Total Vol MK Test')[2])
#         TotalVol_tau.append(value.get('Total Vol MK Test')[4])
        
#         SummerVol_trend.append(value.get('Summer Vol')[0])
#         SummerVol_p.append(value.get('Summer Vol')[2])
#         SummerVol_tau.append(value.get('Summer Vol')[4])
    
#         RunoffVol_trend.append(value.get('Runoff Season Vol')[0])
#         RunoffVol_p.append(value.get('Runoff Season Vol')[2])
#         RunoffVol_tau.append(value.get('Runoff Season Vol')[4])
        
#         WinterVol_trend.append(value.get('Winter Vol')[0])
#         WinterVol_p.append(value.get('Winter Vol')[2])
#         WinterVol_tau.append(value.get('Winter Vol')[4])
    
#         FallVol_trend.append(value.get('Fall Vol')[0])
#         FallVol_p.append(value.get('Fall Vol')[2])
#         FallVol_tau.append(value.get('Fall Vol')[4])
        PeakRunoffQ_trend.append(value.get('Q on date of peak Q date MK Test')[0])
        PeakRunoffQ_p.append(value.get('Q on date of peak Q date MK Test')[2])
        PeakRunoffQ_tau.append(value.get('Q on date of peak Q date MK Test')[4])
        
        PeakRunoffDOY_trend.append(value.get('Q on date of peak Q date MK Test')[0])
        PeakRunoffDOY_p.append(value.get('Q on date of peak Q date MK Test')[2])
        PeakRunoffDOY_tau.append(value.get('Q on date of peak Q date MK Test')[4])    
            
        ThresholdVol_trend.append(value.get('Threshold Vol MK Test')[0])
        ThresholdVol_p.append(value.get('Threshold Vol MK Test')[2])
        ThresholdVol_tau.append(value.get('Threshold Vol MK Test')[4])    
            
        ThresholdDOY_trend.append(value.get('Threshold Vol DOY MK Test')[0])
        ThresholdDOY_p.append(value.get('Threshold Vol DOY MK Test')[2])
        ThresholdDOY_tau.append(value.get('Threshold Vol DOY MK Test')[4])            
            
        TotalVol_trend.append(value.get('Total Vol MK Test')[0])
        TotalVol_p.append(value.get('Total Vol MK Test')[2])
        TotalVol_tau.append(value.get('Total Vol MK Test')[4])
        
        SummerVol_trend.append(value.get('Summer Vol')[0])
        SummerVol_p.append(value.get('Summer Vol')[2])
        SummerVol_tau.append(value.get('Summer Vol')[4])
    
        RunoffVol_trend.append(value.get('Runoff Season Vol')[0])
        RunoffVol_p.append(value.get('Runoff Season Vol')[2])
        RunoffVol_tau.append(value.get('Runoff Season Vol')[4])
        
        WinterVol_trend.append(value.get('Winter Vol')[0])
        WinterVol_p.append(value.get('Winter Vol')[2])
        WinterVol_tau.append(value.get('Winter Vol')[4])
    
        FallVol_trend.append(value.get('Fall Vol')[0])
        FallVol_p.append(value.get('Fall Vol')[2])
        FallVol_tau.append(value.get('Fall Vol')[4])        
        
#     for site, value in volume_stats_mf.items():
        
#         sites.append(site)
#         site_name.append(value.get('name'))
#         site_type.append('modified flows')        
#         lat.append(value.get('lat'))
#         long.append(value.get('long'))              
                         
#         # basins.append(site[1]['Basin'])
        
#         PeakRunoffQ_trend.append(value.get('Q on date of peak Q date MK Test')[0])
#         PeakRunoffQ_p.append(value.get('Q on date of peak Q date MK Test')[2])
#         PeakRunoffQ_tau.append(value.get('Q on date of peak Q date MK Test')[4])
        
#         PeakRunoffDOY_trend.append(value.get('Q on date of peak Q date MK Test')[0])
#         PeakRunoffDOY_p.append(value.get('Q on date of peak Q date MK Test')[2])
#         PeakRunoffDOY_tau.append(value.get('Q on date of peak Q date MK Test')[4])    
            
#         ThresholdVol_trend.append(value.get('Threshold Vol MK Test')[0])
#         ThresholdVol_p.append(value.get('Threshold Vol MK Test')[2])
#         ThresholdVol_tau.append(value.get('Threshold Vol MK Test')[4])    
            
#         ThresholdDOY_trend.append(value.get('Threshold Vol DOY MK Test')[0])
#         ThresholdDOY_p.append(value.get('Threshold Vol DOY MK Test')[2])
#         ThresholdDOY_tau.append(value.get('Threshold Vol DOY MK Test')[4])            
            
#         TotalVol_trend.append(value.get('Total Vol MK Test')[0])
#         TotalVol_p.append(value.get('Total Vol MK Test')[2])
#         TotalVol_tau.append(value.get('Total Vol MK Test')[4])
        
#         SummerVol_trend.append(value.get('Summer Vol')[0])
#         SummerVol_p.append(value.get('Summer Vol')[2])
#         SummerVol_tau.append(value.get('Summer Vol')[4])
    
#         RunoffVol_trend.append(value.get('Runoff Season Vol')[0])
#         RunoffVol_p.append(value.get('Runoff Season Vol')[2])
#         RunoffVol_tau.append(value.get('Runoff Season Vol')[4])
        
#         WinterVol_trend.append(value.get('Winter Vol')[0])
#         WinterVol_p.append(value.get('Winter Vol')[2])
#         WinterVol_tau.append(value.get('Winter Vol')[4])
    
#         FallVol_trend.append(value.get('Fall Vol')[0])
#         FallVol_p.append(value.get('Fall Vol')[2])
#         FallVol_tau.append(value.get('Fall Vol')[4])
    
    peak_runoffQ = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type, 'lat': lat, 'long': long, 'trend': PeakRunoffQ_trend,  'p': PeakRunoffQ_p, 'tau': PeakRunoffQ_tau})
    peak_runoffDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': PeakRunoffDOY_trend,  'p': PeakRunoffDOY_p, 'tau': PeakRunoffDOY_tau})
    ThresholdVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': ThresholdVol_trend, 'p': ThresholdVol_p, 'tau': ThresholdVol_tau})
    ThresholdDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': ThresholdDOY_trend, 'p': ThresholdDOY_p, 'tau': ThresholdDOY_tau})
    total_vol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': TotalVol_trend, 'p': TotalVol_p, 'tau': TotalVol_tau})
    RunoffVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': RunoffVol_trend, 'p': RunoffVol_p, 'tau': RunoffVol_tau})
    SummerVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'site_type': site_type,'lat': lat, 'long': long, 'trend': SummerVol_trend, 'p': SummerVol_p, 'tau': SummerVol_tau})
    WinterVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': WinterVol_trend, 'p': WinterVol_p, 'tau': WinterVol_tau})
    FallVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type, 'lat': lat, 'long': long, 'trend': FallVol_trend, 'p': FallVol_p, 'tau': FallVol_tau})
    
    peak_runoffQ_title = 'Trend in Peak Runoff Volume: Mann-Kendall Test Results' 
    peak_runoffDOY_title = 'Trend in Timing During Peak Runoff: Mann-Kendall Test Results'
    ThresholdDOY_title = 'Trends in Timing of 50% Yearly Volume: Mann-Kendall Test Results'
    ThresholdVol_title = 'Trend in Volume at 50% Yearly Flow: Mann-Kendall Test Results'
    total_vol_title = 'Yearly Total Volume Trends: Mann-Kendall Test Results'
    SummerVol_title = 'Trend in Late Summer Volumes (08-01 through 09-30): Mann-Kendall Test Results'
    FallVol_title = 'Trend in Fall to early Winter Volumes (10-01 through 12-31): Mann-Kendall Test Results'
    WinterVol_title = 'Trend in Wintertime Volumes (01-01 through 4-01): Mann-Kendall Test Results'
    RunoffVol_title = 'Trend in Runoff Season Volumes (04-01 through 07-31): Mann-Kendall Test Results'

Site 12301250 is missing more than 15% of data between 1968-10-01 and 2023-10-01.  It will NOT be included in the study.
Site 12302055 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 12304500 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 12306500 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 12324400 is missing more than 15% of data between 1968-10-01 and 2023-10-01.  It will NOT be included in the study.
Site 12324590 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 12332000 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 12335100 is missing more than 15% of data between 1968-10-01 and 2023-10-01.  It will NOT be included in the study.
Site 12342500 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be includ

Site 08GA071 is missing more than 15% of data between 1968-10-01 and 2023-10-01.  It will NOT be included in the study.
Site 08GA072 is missing more than 15% of data between 1968-10-01 and 2023-10-01.  It will NOT be included in the study.
Site 08HA001 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 08JD006 is missing more than 15% of data between 1968-10-01 and 2023-10-01.  It will NOT be included in the study.
Site 08KA001 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 08KA004 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 08KA005 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 08KA007 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the study.
Site 08KB001 has more than 85% of data between 1968-10-01 and 2023-10-01 and will be included in the

In [27]:
d = dataloader.DataLoader(filtered_all_ca_site_data.get(site), 'Date', name_of_Q_column='Value/Valeur')

AttributeError: 'DataLoader' object has no attribute '_df'

In [62]:
type(filtered_all_ca_site_data.get('07EA005'))

NoneType

In [63]:
all_ca_site_data.get('07EA005')

In [59]:
filtered_all_ca_site_data.get('site') [filtered_all_ca_site_data[' ID']=='07EE009']

In [8]:
peak_runoffQ

,sites,site_name,site_type,lat,long,trend,p,tau
0,12302055,Fisher River near Libby MT,headwater,48.355603,-115.31465,no trend,1.023404e-01,-0.152189
1,12304500,Yaak River near Troy MT,headwater,48.561722,-115.970158,decreasing,4.134116e-02,-0.189899
2,12306500,MOYIE RIVER AT EASTPORT ID,headwater,48.999167,-116.179722,no trend,1.794617e-01,-0.122807
3,12324590,Little Blackfoot River near Garrison MT ST,headwater,46.519483,-112.793172,no trend,6.989881e-01,0.037707
4,12332000,Middle Fork Rock Cr nr Philipsburg MT ST,headwater,46.184569,-113.501569,no trend,8.155748e-01,-0.022078
...,...,...,...,...,...,...,...,...
76,08NK018,53 FORDING RIVER AT THE MOUTH Name: STATION...,canadian,"53 49.893959 Name: LATITUDE, dtype: float64","53 -114.866661 Name: LONGITUDE, dtype: float64",no trend,1.134983e-01,-0.147475
77,08NL004,54 ASHNOLA RIVER NEAR KEREMEOS Name: STATIO...,canadian,"54 49.20763 Name: LATITUDE, dtype: float64","54 -119.993523 Name: LONGITUDE, dtype: float64",no trend,9.075316e-01,0.011448
78,08NN002,55 GRANBY RIVER AT GRAND FORKS Name: STATIO...,canadian,"55 49.04327 Name: LATITUDE, dtype: float64","55 -118.44043 Name: LONGITUDE, dtype: float64",no trend,8.219298e-01,0.021549
79,09AA006,57 ATLIN RIVER NEAR ATLIN Name: STATION_NAM...,canadian,"57 59.59536 Name: LATITUDE, dtype: float64","57 -133.814423 Name: LONGITUDE, dtype: float64",increasing,3.359350e-07,0.381428


In [12]:
figs = []
small_circle = 8
large_circle = 13
#      dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 
for stat in [peak_runoffQ, peak_runoffDOY, ThresholdVol, ThresholdDOY, total_vol, RunoffVol, SummerVol, FallVol, WinterVol]:
    if stat is peak_runoffQ:
        title=peak_runoffQ_title
    elif stat is peak_runoffDOY:
        title=peak_runoffDOY_title
    elif stat is ThresholdVol:
        title=ThresholdVol_title
    elif stat is ThresholdDOY:
        title=ThresholdDOY_title
    elif stat is total_vol:
        title=total_vol_title
    elif stat is RunoffVol:
        title=RunoffVol_title
    elif stat is SummerVol:
        title=SummerVol_title
    elif stat is WinterVol:
        title=WinterVol_title
    elif stat is FallVol:
        title=FallVol_title

#     trend_mapping = {'decreasing': 0, 'no trend': 1, 'increasing': 2}
#     stat = FallVol#, WinterVol, peak_runoffQ
#     stat['trend_numeric'] = stat['trend'].map(trend_mapping)

#         color_scale_decr = [
#             [0.0, 'rgb(128,128,128)'],
#             [0.85, 'rgb(128,128,128)'],
#             [0.851, 'rgb(254,224,144)'],
#             [0.9, 'rgb(253,174,97)'],
#             [.95, 'rgb(215, 48, 39)'],
#             [1.0, 'rgb(165,0,38)']
#         ]

#         color_scale_incr = [
#             [0.0, 'rgb(128,128,128)'],
#             [0.85, 'rgb(128,128,128)'],
#             [0.851, 'rgb(239,243,255)'],
#             [0.9, 'rgb(189,215,231)'],
#             [.95, 'rgb(49,130,189)'],
#             [1.0, 'rgb(8,81,156)']
#         ]
    color_scale_decr = [
        [0.0, 'rgb(165,0,38)'],
        [0.05, 'rgb(215, 48, 39)'],
        [0.1, 'rgb(244,109,67)'],
        [0.15, 'rgb(253,174,97)'],
#             [.2, 'rgb(254,224,144)'],
        [.1501, 'rgb(128,128,128)'],
        [1.0, 'rgb(128,128,128)']
    ]

#         #blues
#         color_scale_incr = [
#             [0.0, 'rgb(8,81,156)'],
#             [0.05, 'rgb(49,130,189)'],
#             [0.1, 'rgb(107,174,214)'],
#             [0.15, 'rgb(189,215,231)'],
#             [.2, 'rgb(239,243,255)'],
#             [.21, 'rgb(128,128,128)'],
#             [1.0, 'rgb(128,128,128)']
#         ]
    #greens
    color_scale_incr = [
        [0.0, 'rgb(0, 109, 44)'],
        [0.05, 'rgb(49, 163, 84)'],
        [0.1, 'rgb(116, 196, 118)'],
        [0.15, 'rgb(186, 228, 179)'],
#             [.2, 'rgb(237,248,233)'],
        [.1501, 'rgb(128,128,128)'],
        [1.0, 'rgb(128,128,128)']
    ]


    # Create bins for p values
    bins = np.linspace(0, 1, num=len(color_scale_decr))

    def get_color_decr(p):
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                return color_scale_decr[i][1]
        return color_scale_decr[-1][1]

    def get_color_incr(p):
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                return color_scale_incr[i][1]
        return color_scale_incr[-1][1]

    # Apply color to each record based on p value

    stat['trend_color1'] = stat[stat['tau']<0]['p'].apply(get_color_decr)
    stat['trend_color2'] = stat[stat['tau']>=0]['p'].apply(get_color_incr)
    stat['trend_color'] = stat['trend_color1'].combine_first(stat['trend_color2'])

    stat['confidence_level'] = (1 - stat['p'])*100

    stat_hw = stat[stat['site_type']=='headwater']
    stat_hw_pos_tau = stat_hw[stat_hw["tau"]>0]
    stat_hw_neg_tau = stat_hw[stat_hw["tau"]<=0]

    stat_mf = stat[stat['site_type']=='modified flows']
    stat_mf_pos_tau = stat_mf[stat_mf["tau"]>0]
    stat_mf_neg_tau = stat_mf[stat_mf["tau"]<=0]

    stat_ca = stat[stat['site_type']=='canadian']
    stat_ca_pos_tau = stat_ca[stat_ca["tau"]>0]
    stat_ca_neg_tau = stat_ca[stat_ca["tau"]<=0]

    traces = []


    # decreasing------------------------------------

    trace = go.Scattermapbox(
            mode='markers',
            lon = stat_hw_neg_tau['long'],
            lat = stat_hw_neg_tau['lat'],
    #                 name = stat['trend'],
            text = stat_hw_neg_tau['site_name'],
    #             textposition='top right',
    #             textfont=dict(
    #                 size=20,
    #                 color='black'
            customdata=stat_hw_neg_tau[['sites','p','tau','trend','site_name']],
            hovertemplate =
                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
            name = 'USGS sites - neg trending',
            showlegend=False,
            marker=go.scattermapbox.Marker(
                size=small_circle,
                color=stat_hw_neg_tau['p'],
                colorscale=color_scale_decr,
                cmin=0,
                cmax=1,
                colorbar = dict(title='P Value - Decr Trend', y=0.25, len=0.5),
    #                 showscale=True,
                symbol='circle',
                )
    )

    traces.append(trace)

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_neg_tau['long'],
#             lat = stat_mf_neg_tau['lat'],
#             text = stat_mf_neg_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_neg_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - neg trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color=stat_mf_neg_tau['p'],
#                 colorscale=color_scale_decr,
#                 cmin=0,
#                 cmax=1,
#     #             colorbar = dict(title='p values'),
#     #                 showscale=True,
#                 symbol='circle'
#             )
#     )
#     traces.append(trace)  


    # increasing----------------------------------------------   

    trace = go.Scattermapbox(
            mode='markers',
            lon = stat_hw_pos_tau['long'],
            lat = stat_hw_pos_tau['lat'],
    #                 name = stat['trend'],
            text = stat_hw_pos_tau['site_name'],
    #             textposition='top right',
    #             textfont=dict(
    #                 size=20,
    #                 color='black'
            customdata=stat_hw_pos_tau[['sites','p','tau','trend','site_name']],
            hovertemplate = 
               '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
            name = 'USGS sites - pos trending',
            showlegend=False,
            marker=go.scattermapbox.Marker(
                size=small_circle,
                color=stat_hw_pos_tau['p'],
                colorscale=color_scale_incr,
                cmin=0,
                cmax=1,
                colorbar = dict(title='P Value - Incr Trend', y=0.75, len=0.5),
    #                 showscale=True,
                symbol='circle',
            )
    )
    traces.append(trace)


#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_pos_tau['long'],
#             lat = stat_mf_pos_tau['lat'],
#     #                 name = stat['trend'],
#             text = stat_mf_pos_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_pos_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - pos trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color=stat_mf_pos_tau['p'],
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#     #                 showscale=True,
#                 symbol='circle',

#         ))
#     traces.append(trace) 






# decreasing------------------------------------
#     print(stat_ca_neg_tau)
    trace = go.Scattermapbox(
        mode='markers',
        lon = [float(long.iloc[0]) for long in stat_ca_neg_tau['long']], #stat_ca_neg_tau['long'],
        lat = [float(lat.iloc[0]) for lat in stat_ca_neg_tau['lat']], #stat_ca_neg_tau['lat'],
#                 name = stat['trend'],
        text = stat_ca_neg_tau['site_name'],
#             textposition='top right',
#             textfont=dict(
#                 size=20,
#                 color='black'
        customdata=stat_ca_neg_tau[['sites','p','tau','trend','site_name']],
        hovertemplate =
            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
        name = 'Canadian sites - neg trending',
        showlegend=False,
        marker=go.scattermapbox.Marker(
            size=small_circle,
            color=stat_ca_neg_tau['p'],
            colorscale=color_scale_decr,
            cmin=0,
            cmax=1,
#             colorbar = dict(title='P Value - Decr Trend', y=0.25, len=0.5),
#                 showscale=True,
            symbol='circle',
            )
    )

    traces.append(trace)



# increasing----------------------------------------------   

    trace = go.Scattermapbox(
        mode='markers',
        lon = [float(long.iloc[0]) for long in stat_ca_pos_tau['long']], #stat_ca_pos_tau['long'],
        lat = [float(lat.iloc[0]) for lat in stat_ca_pos_tau['lat']],
        text = stat_ca_pos_tau['site_name'],
#             textposition='top right',
#             textfont=dict(
#                 size=10,
#                 color='black'
        customdata=stat_ca_pos_tau[['sites','p','tau','trend','site_name']],
        hovertemplate = 
           '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
        name = 'Canadian sites - pos trending',
        showlegend=False,
        marker=go.scattermapbox.Marker(
            size=small_circle,
            color=stat_ca_pos_tau['p'],
            colorscale=color_scale_incr,
            cmin=0,
            cmax=1,
#             colorbar = dict(title='p values'),
#                 showscale=True,
            symbol='circle',
        )
    )
    traces.append(trace)


    #--------- purely to get grey symbols on legend ----------
    #Just ploting one single point that is without a trend (p>0.6) and then using these traces for the legend.

    stat_hw_nuetral = stat_hw[stat_hw["p"]>.6].iloc[0,:]
    trace = go.Scattermapbox(
            mode='markers',
            lon = [stat_hw_nuetral['long']],
            lat = [stat_hw_nuetral['lat']],
    #                 name = stat['trend'],
            text = [stat_hw_nuetral['site_name']],
    #             textposition='top right',
    #             textfont=dict(
    #                 size=20,
    #                 color='black'
            customdata=stat_hw_nuetral[['sites','p','tau','trend','site_name']],
            hovertemplate = 
               '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
            name = 'USGS and EC Sites',
            showlegend=True,
            marker=go.scattermapbox.Marker(
                size=small_circle,
                color='grey',
                colorscale=color_scale_incr,
                cmin=0,
                cmax=1,
                symbol='circle',
        ))
    traces.append(trace) 

#     stat_mf_nuetral = stat_mf[stat_mf["p"]>.6].iloc[0,:]
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = [stat_hw_nuetral['long']],
#             lat = [stat_hw_nuetral['lat']],
#     #                 name = stat['trend'],
#             text = [stat_hw_nuetral['site_name']],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_nuetral[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows',
#             showlegend=True,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color='grey',
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 symbol='circle',
#         ))
#     traces.append(trace) 


    #------------

    layout = go.Layout(
        mapbox_layers=[
            {
                "below": 'traces',
                "sourcetype": "raster",
                "sourceattribution": "United States Geological Survey",
                "source": [
                    "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
                ],
            }
        ],        
        mapbox = {
    #         'accesstoken': '',
    #         'style': "stamen-terrain",
            'style': 'carto-positron',#'outdoors', #'stamen-terrain',  
            'zoom':4.5,
            'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
        },
        title=f'<b>{title}</b>',
        showlegend=True,
        legend=dict(x=0.5, y=0, traceorder='normal', orientation='h'),
        autosize=True,
        width=1100,
        height=900
    )

    fig = go.Figure(data=traces, layout=layout)
    fig.show()
    figs.append(fig)



file_name = f'mk_maps_usgs_ca_sites{start_date}to{end_date}.html'
if os.path.exists(file_name):
    os.remove(file_name)
for figure in figs:
    with open(file_name, 'a') as f:
        f.write(figure.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))                        

In [ ]:
stat_ca

In [ ]:
traces

In [ ]:
pd.DataFrame(volume_stats_mf)

In [ ]:
# figs = []
# #      dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 
# for stat in [peak_runoffQ, peak_runoffDOY, ThresholdVol, ThresholdDOY, total_vol, RunoffVol, SummerVol, FallVol, WinterVol]:
#     if stat is peak_runoffQ:
#         title=peak_runoffQ_title
#     elif stat is peak_runoffDOY:
#         title=peak_runoffDOY_title
#     elif stat is ThresholdVol:
#         title=ThresholdVol_title
#     elif stat is ThresholdDOY:
#         title=ThresholdDOY_title
#     elif stat is total_vol:
#         title=total_vol_title
#     elif stat is RunoffVol:
#         title=RunoffVol_title
#     elif stat is SummerVol:
#         title=SummerVol_title
#     elif stat is WinterVol:
#         title=WinterVol_title
#     elif stat is FallVol:
#         title=FallVol_title

# #     stat = FallVol#, WinterVol, peak_runoffQ
# #     stat['trend_numeric'] = stat['trend'].map(trend_mapping)

#     color_scale_decr = [
#         [0.0, 'rgb(165,0,38)'],
#         [0.05, 'rgb(215, 48, 39)'],
#         [0.1, 'rgb(244,109,67)'],
#         [0.15, 'rgb(253,174,97)'],
#         [.2, 'rgb(254,224,144)'],
#         [.21, 'rgb(128,128,128)'],
#         [1.0, 'rgb(128,128,128)']
#     ]

#     color_scale_incr = [
#         [0.0, 'rgb(8,81,156)'],
#         [0.05, 'rgb(49,130,189)'],
#         [0.1, 'rgb(107,174,214)'],
#         [0.15, 'rgb(189,215,231)'],
#         [.2, 'rgb(239,243,255)'],
#         [.21, 'rgb(128,128,128)'],
#         [1.0, 'rgb(128,128,128)']
#     ]


#     # Create bins for p values
#     bins = np.linspace(0, 1, num=len(color_scale_decr))

#     def get_color_decr(p):
#         for i in range(len(bins) - 1):
#             if bins[i] <= p < bins[i + 1]:
#                 return color_scale_decr[i][1]
#         return color_scale_decr[-1][1]

#     def get_color_incr(p):
#         for i in range(len(bins) - 1):
#             if bins[i] <= p < bins[i + 1]:
#                 return color_scale_incr[i][1]
#         return color_scale_incr[-1][1]

#     # Apply color to each record based on p value

#     stat['trend_color1'] = stat[stat['tau']<0]['p'].apply(get_color_decr)
#     stat['trend_color2'] = stat[stat['tau']>=0]['p'].apply(get_color_incr)
#     stat['trend_color'] = stat['trend_color1'].combine_first(stat['trend_color2'])

#     stat_hw = stat[stat['site_type']=='headwater']
#     stat_hw_pos_tau = stat_hw[stat_hw["tau"]>0]
#     stat_hw_neg_tau = stat_hw[stat_hw["tau"]<=0]

#     stat_mf = stat[stat['site_type']=='modified flows']
#     stat_mf_pos_tau = stat_mf[stat_mf["tau"]>0]
#     stat_mf_neg_tau = stat_mf[stat_mf["tau"]<=0]

#     # trends = ['decreasing', 'no trend', 'increasing']
#     # trends_numerics = [0, 1, 2]
#     # markers = ['circle', 'circle-stroked', 'information']
#     # colors = ['red', 'grey', 'blue']
#     # site_types = ['modified flows', 'headwater']
#     # symbol_map = {'modified flows': 'square', 'headwater': 'circle'}

#     traces = []


#     # decreasing------------------------------------

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_hw_neg_tau['long'],
#             lat = stat_hw_neg_tau['lat'],
#     #                 name = stat['trend'],
#             text = stat_hw_neg_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_neg_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate =
#                 '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
#             name = 'USGS sites - neg trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=7,
#                 color=stat_hw_neg_tau['p'],
#                 colorscale=color_scale_decr,
#                 cmin=0,
#                 cmax=1,
#                 colorbar = dict(title='p values - Decr Trend', y=0.25, len=0.5),
#     #                 showscale=True,
#                 symbol='circle',
#                 )
#     )

#     traces.append(trace)

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_neg_tau['long'],
#             lat = stat_mf_neg_tau['lat'],
#             text = stat_mf_neg_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_neg_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - neg trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=10,
#                 color=stat_mf_neg_tau['p'],
#                 colorscale=color_scale_decr,
#                 cmin=0,
#                 cmax=1,
#     #             colorbar = dict(title='p values'),
#     #                 showscale=True,
#                 symbol='circle'
#             )
#     )
#     traces.append(trace)  


#     # increasing----------------------------------------------   

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_hw_pos_tau['long'],
#             lat = stat_hw_pos_tau['lat'],
#     #                 name = stat['trend'],
#             text = stat_hw_pos_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_pos_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'USGS sites - pos trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=6,
#                 color=stat_hw_pos_tau['p'],
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#     #             colorbar = dict(title='p values'),
#     #                 showscale=True,
#                 symbol='circle',
#             )
#     )
#     traces.append(trace)


#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_pos_tau['long'],
#             lat = stat_mf_pos_tau['lat'],
#     #                 name = stat['trend'],
#             text = stat_mf_pos_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_pos_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - pos trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=11,
#                 color=stat_mf_pos_tau['p'],
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 colorbar = dict(title='p values - Incr Trend', len=0.5,y=.77),
#     #                 showscale=True,
#                 symbol='circle',

#         ))
#     traces.append(trace) 


#     #--------- purely to get grey symbols on legend ----------
#     #Just ploting one single point that is without a trend (p>0.6) and then using these traces for the legend.

#     stat_hw_nuetral = stat_hw[stat_hw["p"]>.6].iloc[0,:]
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = [stat_hw_nuetral['long']],
#             lat = [stat_hw_nuetral['lat']],
#     #                 name = stat['trend'],
#             text = [stat_hw_nuetral['site_name']],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_nuetral[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'USGS Sites',
#             showlegend=True,
#             marker=go.scattermapbox.Marker(
#                 size=5,
#                 color='grey',
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 symbol='circle',
#         ))
#     traces.append(trace) 

#     stat_mf_nuetral = stat_mf[stat_mf["p"]>.6].iloc[0,:]
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = [stat_mf_nuetral['long']],
#             lat = [stat_mf_nuetral['lat']],
#     #                 name = stat['trend'],
#             text = [stat_mf_nuetral['site_name']],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_nuetral[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows',
#             showlegend=True,
#             marker=go.scattermapbox.Marker(
#                 size=11,
#                 color='grey',
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 symbol='circle',
#         ))
#     traces.append(trace) 


#     #------------

#     layout = go.Layout(
#         mapbox_layers=[
#             {
#                 "below": 'traces',
#                 "sourcetype": "raster",
#                 "sourceattribution": "United States Geological Survey",
#                 "source": [
#                     "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
#                 ],
#             }
#         ],        
#         mapbox = {
#     #         'accesstoken': '',
#     #         'style': "stamen-terrain",
#             'style': 'carto-positron',#'outdoors', #'stamen-terrain',  
#             'zoom':5,
#             'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
#         },
#         title=f'<b>{title}</b>',
#         showlegend=True,
#         legend=dict(x=0.5, y=-0.1, traceorder='normal', orientation='h'),
#         autosize=True,
#         width=900,
#         height=700
#     )

#     fig = go.Figure(data=traces, layout=layout)
#     fig.show()
#     figs.append(fig)

In [ ]:
# ## Works well but doesn't distinguish between increasing and decreasing trends.  Awesome scale bar for p values though.

# trend_mapping = {'decreasing': 0, 'no trend': 1, 'increasing': 2}


# stat = WinterVol #peak_runoffQ
# stat['trend_numeric'] = stat['trend'].map(trend_mapping)

# # color_scale = [
# #     [0.0, 'rgb(165,0,38)'],
# #     [0.1, 'rgb(215,48,39)'],
# #     [0.2, 'rgb(244,109,67)'],
# #     [0.3, 'rgb(253,174,97)'],
# #     [0.4, 'rgb(254,224,144)'],
# #     [0.5, 'rgb(224,243,248)'],
# #     [0.6, 'rgb(171,217,233)'],
# #     [0.7, 'rgb(116,173,209)'],
# #     [0.8, 'rgb(69,117,180)'],
# #     [0.9, 'rgb(49,54,149)'],
# #     [1.0, 'rgb(2,56,88)']
# # ]
# color_scale = [
#     [0.0, 'rgb(165,0,38)'],
#     [0.05, 'rgb(215, 48, 39)'],
#     [0.1, 'rgb(244,109,67)'],
#     [0.15, 'rgb(253,174,97)'],
#     [.2, 'rgb(254,224,144)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]

# # Create bins for p values
# bins = np.linspace(0, 1, num=len(color_scale))

# def get_color(p):
#     for i in range(len(bins) - 1):
#         if bins[i] <= p < bins[i + 1]:
#             return color_scale[i][1]
#     return color_scale[-1][1]

# # Apply color to each record based on p value
# stat['trend_color'] = stat['p'].apply(get_color)
# stat_hw = peak_runoffQ[peak_runoffQ['site_type']=='headwater']
# stat_mf = peak_runoffQ[peak_runoffQ['site_type']=='modified flows']


# trends = ['decreasing', 'no trend', 'increasing']
# trends_numerics = [0, 1, 2]
# markers = ['circle', 'circle-stroked', 'information']
# colors = ['red', 'grey', 'blue']
# site_types = ['modified flows', 'headwater']
# symbol_map = {'modified flows': 'square', 'headwater': 'circle'}

# traces = []
# # fig = go.Figure()
# for trend, color in zip(trends, colors):
# # for site_type, group in stat.groupby('site_type'):
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_hw[stat_hw["trend"]==trend]['long'],
#             lat = stat_hw[stat_hw["trend"]==trend]['lat'],
# #                 name = stat['trend'],
#             text = stat_hw[stat_hw["trend"]==trend]['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#             customdata=stat_hw[stat_hw["trend"]==trend][['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#             marker=go.scattermapbox.Marker(
#                 size=10,
#                 color=stat_hw[stat_hw["trend"]==trend]['p'],
#                 colorscale=color_scale,
#                 cmin=0,
#                 cmax=1,
#                 colorbar = dict(title='p values'),
# #                 showscale=True,
#                 symbol='circle', 
#         ))
#     traces.append(trace)


# for trend, color in zip(trends, colors):
# # for site_type, group in stat.groupby('site_type'):
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf[stat_mf["trend"]==trend]['long'],
#             lat = stat_mf[stat_mf["trend"]==trend]['lat'],
# #                 name = stat['trend'],
#             text = stat_mf[stat_mf["trend"]==trend]['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#             customdata=stat_mf[stat_mf["trend"]==trend][['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#             marker=go.scattermapbox.Marker(
#                 size=10,
#                 color=stat_mf[stat_mf["trend"]==trend]['p'],
#                 colorscale=color_scale,
#                 cmin=0,
#                 cmax=1,
#                 colorbar = dict(title='p values'),
# #                 showscale=True,
#                 symbol='circle', 
#         ))
#     traces.append(trace)
    
    
# layout = go.Layout(
# # fig.update_layout(

# #     mapbox_layers=[
# #         {
# #             "below": 'traces',
# #             "sourcetype": "raster",
# #             "sourceattribution": "United States Geological Survey",
# #             "source": [
# #                 "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
# #             ],
# #         }
# #     ],        
#     mapbox = {
#         'accesstoken': '',
# #         'style': "stamen-terrain",
#         'style': 'outdoors',#'carto-positron', #'stamen-terrain',  
#         'zoom':5,
#         'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
#     },
#     title=f'<b>{title}</b>',
#     showlegend=False,
#     autosize=True,
#     width=900,
#     height=700
# )

# fig = go.Figure(data=traces, layout=layout)
# fig.show()

In [ ]:
# trend_mapping = {'decreasing': 0, 'no trend': 1, 'increasing': 2}


# stat = FallVol#, WinterVol, peak_runoffQ
# stat['trend_numeric'] = stat['trend'].map(trend_mapping)

# # color_scale = [
# #     [0.0, 'rgb(165,0,38)'],
# #     [0.1, 'rgb(215,48,39)'],
# #     [0.2, 'rgb(244,109,67)'],
# #     [0.3, 'rgb(253,174,97)'],
# #     [0.4, 'rgb(254,224,144)'],
# #     [0.5, 'rgb(224,243,248)'],
# #     [0.6, 'rgb(171,217,233)'],
# #     [0.7, 'rgb(116,173,209)'],
# #     [0.8, 'rgb(69,117,180)'],
# #     [0.9, 'rgb(49,54,149)'],
# #     [1.0, 'rgb(2,56,88)']
# # ]
# color_scale_decr = [
#     [0.0, 'rgb(165,0,38)'],
#     [0.05, 'rgb(215, 48, 39)'],
#     [0.1, 'rgb(244,109,67)'],
#     [0.15, 'rgb(253,174,97)'],
#     [.2, 'rgb(254,224,144)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]

# color_scale_incr = [
#     [0.0, 'rgb(8,81,156)'],
#     [0.05, 'rgb(49,130,189)'],
#     [0.1, 'rgb(107,174,214)'],
#     [0.15, 'rgb(189,215,231)'],
#     [.2, 'rgb(239,243,255)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]


# # Create bins for p values
# bins = np.linspace(0, 1, num=len(color_scale))

# def get_color_decr(p):
#     for i in range(len(bins) - 1):
#         if bins[i] <= p < bins[i + 1]:
#             return color_scale_decr[i][1]
#     return color_scale_decr[-1][1]

# def get_color_incr(p):
#     for i in range(len(bins) - 1):
#         if bins[i] <= p < bins[i + 1]:
#             return color_scale_incr[i][1]
#     return color_scale_incr[-1][1]

# # Apply color to each record based on p value

# stat['trend_color'] = stat[stat['trend']=='decreasing']['p'].apply(get_color_decr)
# stat['trend_color'] = stat[stat['trend']=='increasing']['p'].apply(get_color_incr)

# stat_hw = stat[stat['site_type']=='headwater']
# stat_mf = stat[stat['site_type']=='modified flows']


# trends = ['decreasing', 'no trend', 'increasing']
# trends_numerics = [0, 1, 2]
# markers = ['circle', 'circle-stroked', 'information']
# colors = ['red', 'grey', 'blue']
# site_types = ['modified flows', 'headwater']
# symbol_map = {'modified flows': 'square', 'headwater': 'circle'}

# traces = []


# # decreasing------------------------------------

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_hw[stat_hw["trend"]=='decreasing']['long'],
#         lat = stat_hw[stat_hw["trend"]=='decreasing']['lat'],
# #                 name = stat['trend'],
#         text = stat_hw[stat_hw["trend"]=='decreasing']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw[stat_hw["trend"]=='decreasing'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#        '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_hw[stat_hw["trend"]=='decreasing']['p'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='p values - Decr Trend', y=0.25, len=0.5),
# #                 showscale=True,
#             symbol='circle', 
#     ))
# traces.append(trace)

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf[stat_mf["trend"]=='decreasing']['long'],
#         lat = stat_mf[stat_mf["trend"]=='decreasing']['lat'],
# #                 name = stat['trend'],
#         text = stat_mf[stat_mf["trend"]=='decreasing']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf[stat_mf["trend"]=='decreasing'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_mf[stat_mf["trend"]=='decreasing']['p'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
# #             colorbar = dict(title='p values'),
# #                 showscale=True,
#             symbol='circle', 
#     ))
# traces.append(trace)  


# # no trend---------------------------------------------------

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_hw[stat_hw["trend"]=='no trend']['long'],
#         lat = stat_hw[stat_hw["trend"]=='no trend']['lat'],
# #                 name = stat['trend'],
#         text = stat_hw[stat_hw["trend"]=='no trend']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw[stat_hw["trend"]=='no trend'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color='rgb(128,128,128)',
#             symbol='circle', 
#     ))
# traces.append(trace)

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf[stat_mf["trend"]=='no trend']['long'],
#         lat = stat_mf[stat_mf["trend"]=='no trend']['lat'],
# #                 name = stat['trend'],
#         text = stat_mf[stat_mf["trend"]=='no trend']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf[stat_mf["trend"]=='no trend'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color='rgb(128,128,128)',
#             symbol='circle', 
#     ))
# traces.append(trace)
 
    
# # increasing----------------------------------------------   
  
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_hw[stat_hw["trend"]=='increasing']['long'],
#         lat = stat_hw[stat_hw["trend"]=='increasing']['lat'],
# #                 name = stat['trend'],
#         text = stat_hw[stat_hw["trend"]=='increasing']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw[stat_hw["trend"]=='increasing'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_hw[stat_hw["trend"]=='increasing']['p'],
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
# #             colorbar = dict(title='p values'),
# #                 showscale=True,
#             symbol='circle', 
#     ))
# traces.append(trace)


# # for trend, color in zip(trends, colors):
# # for site_type, group in stat.groupby('site_type'):
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf[stat_mf["trend"]=='increasing']['long'],
#         lat = stat_mf[stat_mf["trend"]=='increasing']['lat'],
# #                 name = stat['trend'],
#         text = stat_mf[stat_mf["trend"]=='increasing']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf[stat_mf["trend"]=='increasing'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_mf[stat_mf["trend"]=='increasing']['p'],
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='p values - Incr Trend', len=0.5,y=.77),
# #                 showscale=True,
#             symbol='circle', 
#     ))
# traces.append(trace) 


    
    
    
    
    
    
    
    
# layout = go.Layout(
# # fig.update_layout(

#     mapbox_layers=[
#         {
#             "below": 'traces',
#             "sourcetype": "raster",
#             "sourceattribution": "United States Geological Survey",
#             "source": [
#                 "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
#             ],
#         }
#     ],        
#     mapbox = {
# #         'accesstoken': '',
# #         'style': "stamen-terrain",
#         'style': 'carto-positron',#'outdoors', #'stamen-terrain',  
#         'zoom':5,
#         'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
#     },
#     title=f'<b>{title}</b>',
#     showlegend=False,
#     autosize=True,
#     width=900,
#     height=700
# )

# fig = go.Figure(data=traces, layout=layout)
# fig.show()

In [ ]:
trend_mapping = {'decreasing': 0, 'no trend': 1, 'increasing': 2}


stat = peak_runoffQ
stat['trend_numeric'] = stat['trend'].map(trend_mapping)
stat['confidence_level'] = 1 - stat['p']

color_scale_decr = [
    [0.0, 'rgb(128,128,128)'],
    [0.85, 'rgb(128,128,128)'],
    [0.851, 'rgb(254,224,144)'],
    [0.9, 'rgb(253,174,97)'],
    [.95, 'rgb(215, 48, 39)'],
    [1.0, 'rgb(165,0,38)']
]

color_scale_incr = [
    [0.0, 'rgb(128,128,128)'],
    [0.85, 'rgb(128,128,128)'],
    [0.851, 'rgb(239,243,255)'],
    [0.9, 'rgb(189,215,231)'],
    [.95, 'rgb(49,130,189)'],
    [1.0, 'rgb(8,81,156)']
]

# color_scale_incr = [
#     [0.0, 'rgb(8,81,156)'],
#     [0.05, 'rgb(49,130,189)'],
#     [0.1, 'rgb(107,174,214)'],
#     [0.15, 'rgb(189,215,231)'],
#     [.2, 'rgb(239,243,255)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]

# color_scale_decr = [
#     [0.0, 'rgb(165,0,38)'],
#     [0.05, 'rgb(215, 48, 39)'],
#     [0.1, 'rgb(244,109,67)'],
#     [0.15, 'rgb(253,174,97)'],
#     [.2, 'rgb(254,224,144)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]




# Create bins for p values
bins = np.linspace(0, 1, num=len(color_scale_decr))

def get_color_decr(p):
    for i in range(len(bins) - 1):
        if bins[i] <= p < bins[i + 1]:
            return color_scale_decr[i][1]
    return color_scale_decr[-1][1]

def get_color_incr(p):
    for i in range(len(bins) - 1):
        if bins[i] <= p < bins[i + 1]:
            return color_scale_incr[i][1]
    return color_scale_incr[-1][1]

# Apply color to each record based on p value

stat['trend_color1'] = stat[stat['tau']<0]['confidence_level'].apply(get_color_decr)
stat['trend_color2'] = stat[stat['tau']>=0]['confidence_level'].apply(get_color_incr)
stat['trend_color'] = stat['trend_color1'].combine_first(stat['trend_color2'])

stat_hw = stat[stat['site_type']=='headwater']
stat_hw_pos_tau = stat_hw[stat_hw["tau"]>0]
stat_hw_neg_tau = stat_hw[stat_hw["tau"]<=0]

stat_mf = stat[stat['site_type']=='modified flows']
stat_mf_pos_tau = stat_mf[stat_mf["tau"]>0]
stat_mf_neg_tau = stat_mf[stat_mf["tau"]<=0]

# trends = ['decreasing', 'no trend', 'increasing']
# trends_numerics = [0, 1, 2]
# markers = ['circle', 'circle-stroked', 'information']
# colors = ['red', 'grey', 'blue']
# site_types = ['modified flows', 'headwater']
# symbol_map = {'modified flows': 'square', 'headwater': 'circle'}

traces = []


# decreasing------------------------------------

trace = go.Scattermapbox(
        mode='markers',
        lon = stat_hw_neg_tau['long'],
        lat = stat_hw_neg_tau['lat'],
#                 name = stat['trend'],
        text = stat_hw_neg_tau['site_name'],
#             textposition='top right',
#             textfont=dict(
#                 size=20,
#                 color='black'
        customdata=stat_hw_neg_tau[['sites','confidence_level','tau','trend','site_name']],
        hovertemplate =
            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
        name = 'USGS sites - neg trending',
        showlegend=False,
#         marker=go.scattermapbox.Marker(
#             size=7,
#             color=stat_hw_neg_tau['confidence_level'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='Confidence - Decr Trend', y=0.25, len=0.5),
# #                 showscale=True,
#             ),
    #fiiiiiiiiiiii
        marker = {'size':15, 'symbol':['triangle' for i in stat_hw_neg_tau.index], 
#                   'color':stat_hw_neg_tau['confidence_level']
                 },
#         color=stat_hw_neg_tau['confidence_level'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='Confidence - Decr Trend', y=0.25, len=0.5),
# #                 showscale=True,
)
                           
traces.append(trace)

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf_neg_tau['long'],
#         lat = stat_mf_neg_tau['lat'],
#         text = stat_mf_neg_tau['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf_neg_tau[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'Modified Flows - neg trending',
#         showlegend=False,
#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_mf_neg_tau['confidence_level'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
# #             colorbar = dict(title='p values'),
# #                 showscale=True,
#             symbol='circle-stroked'
#         )
# )
# traces.append(trace)  


# # increasing----------------------------------------------   
  
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_hw_pos_tau['long'],
#         lat = stat_hw_pos_tau['lat'],
# #                 name = stat['trend'],
#         text = stat_hw_pos_tau['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw_pos_tau[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'USGS sites - pos trending',
#         showlegend=False,
#         marker=go.scattermapbox.Marker(
#             size=6,
#             color=stat_hw_pos_tau['confidence_level'],
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
# #             colorbar = dict(title='p values'),
# #                 showscale=True,
#             symbol=['square' for i in stat_hw_pos_tau.index],
#         )
# )
# traces.append(trace)


# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf_pos_tau['long'],
#         lat = stat_mf_pos_tau['lat'],
# #                 name = stat['trend'],
#         text = stat_mf_pos_tau['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf_pos_tau[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'Modified Flows - pos trending',
#         showlegend=False,
#         marker=go.scattermapbox.Marker(
#             size=11,
#             color=stat_mf_pos_tau['confidence_level'],
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='Confidence - Incr Trend', len=0.5,y=.77),
# #                 showscale=True,
#             symbol=['square' for i in stat_mf_pos_tau.index],

#     ))
# traces.append(trace) 


#--------- purely to get grey symbols on legend ----------
#Just ploting one single point that is without a trend (p>0.6) and then using these traces for the legend.

# stat_hw_nuetral = stat_hw[stat_hw["p"]>.6].iloc[0,:]
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = [stat_hw_nuetral['long']],
#         lat = [stat_hw_nuetral['lat']],
# #                 name = stat['trend'],
#         text = [stat_hw_nuetral['site_name']],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw_nuetral[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'USGS Sites',
#         showlegend=True,
#         marker=go.scattermapbox.Marker(
#             size=5,
#             color='grey',
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
#             symbol='circle',
#     ))
# traces.append(trace) 

# stat_mf_nuetral = stat_mf[stat_mf["p"]>.6].iloc[0,:]
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = [stat_mf_nuetral['long']],
#         lat = [stat_mf_nuetral['lat']],
# #                 name = stat['trend'],
#         text = [stat_mf_nuetral['site_name']],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf_nuetral[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'Modified Flows',
#         showlegend=True,
#         marker=go.scattermapbox.Marker(
#             size=11,
#             color='grey',
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
#             symbol='circle',
#     ))
# traces.append(trace) 


#------------
  
layout = go.Layout(
#     mapbox_layers=[
#         {
#             "below": 'traces',
#             "sourcetype": "raster",
#             "sourceattribution": "United States Geological Survey",
#             "source": [
#                 "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
#             ],
#         }
#     ],        
    mapbox = {
        'accesstoken': '',
#         'style': "stamen-terrain",
        'style':'outdoors',
#         'style': 'carto-positron',  
        'zoom':4,
        'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
    },
    title=f'<b>{title}</b>',
    showlegend=True,
    legend=dict(x=0.5, y=0, traceorder='normal', orientation = 'h'),
    autosize=True,
    width=900,
    height=700
)

fig = go.Figure(data=traces, layout=layout)
fig.show()

In [ ]:
stat_hw_nuetral

In [ ]:
stat

In [ ]:
stat_hw[stat_hw['sites']==13338500]

In [ ]:
stat_hw[stat_hw["p"]>.6].iloc[0,:]